# FSRS-6 prema GRU na uzorku anki-revlogs-10k (Etapa 11a, portfolio)

Reprodukcija [srs-benchmarka](https://github.com/open-spaced-repetition/srs-benchmark) na 100 kolekcija, postavka „without same-day reviews“. Protokol, razlike prema benchmarku i šta se prenosi na Hoot su u [`ml/fsrs/README.md`](../fsrs/README.md).

Notebook čita samo metrike (`ml/fsrs/results/users.jsonl`), ne podatke. Bez tog fajla prvo pokreni `hoot_ml.srsbench.data` i `hoot_ml.srsbench.bench` (README).

**Upozorenje:** Hoot jedinice su poglavlja i ispitna pitanja sa ocjenama 0/1/2 (Again/Hard/Good), a ne Anki kartice sa četiri dugmeta. Brojevi odavde se ne prenose 1:1 na tačnost FSRS-a u Hootu. To mjeri `hoot_ml.monitor`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown

from hoot_ml.srsbench.bench import REFERENCE, report

results = Path("../fsrs/results/users.jsonl")
users = pd.DataFrame([json.loads(line) for line in results.read_text().splitlines()])
users.groupby("model")["size"].describe()[["count", "50%", "mean", "max"]]

## Tabela

Prosjek po korisnicima ± 99 % CI, u zadnjoj koloni brojevi iz README-a benchmarka (svih 9 999 kolekcija). Ispod je prosjek ponderisan brojem ponavljanja i udio korisnika kod kojih je GRU bolji.

In [ ]:
Markdown(report(users))

## Raspodjela po korisnicima

In [ ]:
metrics = ["log_loss", "rmse_bins", "auc"]
models = list(users["model"].unique())
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, metrics, strict=True):
    ax.boxplot(
        [users.loc[users["model"] == m, metric].dropna() for m in models], tick_labels=models
    )
    for m, ref in REFERENCE.items():
        if m in models:
            ax.scatter(models.index(m) + 1, ref[metric], marker="x", color="red", zorder=3)
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=30)
fig.suptitle("crveni x = benchmark (svi korisnici)")
plt.tight_layout()

## GRU − FSRS-6 po korisniku prema broju ponavljanja

GRU bez pretreninga treba podatke. Ispod nule (log-loss, RMSE) GRU je bolji. Očekivanje: razlika pada sa brojem ponavljanja, a korisnici sa malo ponavljanja (bliže Hootu) ostaju na strani FSRS-6.

In [ ]:
pivot = users.pivot(index="user", columns="model", values=["log_loss", "rmse_bins", "size"])
size = pivot["size"]["fsrs6"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["log_loss", "rmse_bins"], strict=True):
    diff = pivot[metric]["gru"] - pivot[metric]["fsrs6"]
    ax.scatter(size, diff, s=12)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_xscale("log")
    ax.set_xlabel("ponavljanja u testu")
    ax.set_title(f"{metric}: gru − fsrs6")
plt.tight_layout()
small = size < size.median()
pd.DataFrame(
    {
        "gru bolji (log-loss), manja polovina": [
            (pivot["log_loss"]["gru"] < pivot["log_loss"]["fsrs6"])[small].mean()
        ],
        "gru bolji (log-loss), veća polovina": [
            (pivot["log_loss"]["gru"] < pivot["log_loss"]["fsrs6"])[~small].mean()
        ],
    }
).round(2)

## Hoot skala i default težine

`fsrs6_3btn`: Easy pročitan kao Good (Hoot nema Easy). `fsrs6_default`: težine bez ličnog fita, kao Hoot prije ~100 ponavljanja. Razlika prema `fsrs6` je ono što lični parametri donose na Anki podacima.

In [ ]:
lift = pd.DataFrame(
    {
        f"{m} − fsrs6": (
            users[users["model"] == m].set_index("user")[metrics]
            - users[users["model"] == "fsrs6"].set_index("user")[metrics]
        ).mean()
        for m in ["fsrs6_default", "fsrs6_3btn"]
    }
)
lift.round(4)

## Zaključak

Popunjava se iz rezultata. Za Hoot: FSRS-6 sa ličnim parametrima ostaje izbor (21 parametar, default kao prior, radi na stotinama ponavljanja i u browseru). Neuronski model bi imao smisla tek uz pretrening na mnogo korisnika.